# Concurrent programming

Some programmers had a problem. They thought to themselves, “We’ll solve it with threads!” 

have Now problems. two they

## Outline

* Sequential code
* Concurrent code
  * multi-threading
  * multi-processing
  * async (very brief)

Note:
I took much of this from [Raymond Hettinger's talks](https://youtu.be/9zinZmE3Ogk?si=-S3KjLw2kp2X_Nai) about concurrency in python.

## Why concurrency

1. Improve perceived responsiveness
2. Improve speed
3. Because that is how the real world works

## Kinds of concurrency

### Thread

Threads have shared state. 

* This is good because we don't have to copy stuff.
* This is bad because we have to manage race conditions.

### Process

Processes do not share state

* This is bad because we have to copy stuff.
* This is good because we do not have to manage race conditions.

    
 ## [Amdah'ls Law](https://en.wikipedia.org/wiki/Amdahl%27s_law)

 > the overall performance improvement gained by optimizing a single part of a system is limited by the fraction of time that the improved part is actually used.

If 90% of your task is inherently sequential, the parallelizing it won't help much.


## Sequential code

### Easy

In [8]:
counter = 0
print('Starting up')
for i in range(10):
    counter += 1
    print(f'The count is {counter}')
    print('--------------------')
print('Finishing up')

Starting up
The count is 1
--------------------
The count is 2
--------------------
The count is 3
--------------------
The count is 4
--------------------
The count is 5
--------------------
The count is 6
--------------------
The count is 7
--------------------
The count is 8
--------------------
The count is 9
--------------------
The count is 10
--------------------
Finishing up


### Factor re-usable code into functions

(We're professionals)

In [9]:
counter = 0

def worker():
    'My job is to increment the counter and print the current count'
    global counter
    counter += 1
    print(f'The count is {counter}')
    print('--------------------')

print('Starting up')
for i in range(10):
    worker()
    
print('Finishing up')

Starting up
The count is 1
--------------------
The count is 2
--------------------
The count is 3
--------------------
The count is 4
--------------------
The count is 5
--------------------
The count is 6
--------------------
The count is 7
--------------------
The count is 8
--------------------
The count is 9
--------------------
The count is 10
--------------------
Finishing up


# Mult-threading

## Multi-threading is easy!

Change one line of code, and our program is now concurrent.

In [10]:
import threading

counter = 0

def worker():
    'My job is to increment the counter and print the current count'
    global counter
    counter += 1
    print('The count is %d' % counter)
    print('--------------------')

print('Starting up')
for i in range(10):
    threading.Thread(target=worker).start()
print('Finishing up')

Starting up
The count is 1
--------------------
The count is 2
--------------------
The count is 3
--------------------
The count is 4
--------------------
The count is 5
--------------------
The count is 6
--------------------
The count is 7
--------------------
The count is 8
--------------------
The count is 9
--------------------
The count is 10
--------------------
Finishing up


## Testing




> We tested the code, and it produced the correct output.  Therefore it is bug free.

This is NOT TRUE for multi-threaded programs.

### Can you spot the race-condition?

Testing cannot prove the absence of errors. It is still useful, but don't rely on it. Many race conditions don't reveal themselves in test environments.

### Fuzzing

A technique for amplifying race conditions.

In [11]:
import threading, time, random

FUZZ = True
counter = 0

def fuzz():
    if FUZZ:
        time.sleep(random.random())

def worker():
    'My job is to increment the counter and print the current count'
    global counter
    id = threading.get_ident()
    fuzz()
    oldcnt = counter
    fuzz()
    counter = oldcnt + 1
    fuzz()
    print(f'(thread {id})The count is {counter}', end='')
    fuzz()
    print()
    fuzz()
    print(f'(thread {id})--------------------', end='')
    fuzz()
    print(f'(thread {id})')
    fuzz()

print('Starting up')
fuzz()
for i in range(10):
    threading.Thread(target=worker).start()
    fuzz()
fuzz()
print('Finishing up')
fuzz()

Starting up
(thread 124635152627264)The count is 1(thread 124635748214336)The count is 1

(thread 124635186198080)The count is 2(thread 124635748214336)--------------------(thread 124635152627264)--------------------
(thread 124635152627264)
(thread 124635748214336)
(thread 124635186198080)--------------------(thread 124635186198080)
(thread 124635169412672)The count is 4
(thread 124634582218304)The count is 4(thread 124635177805376)The count is 4
(thread 124635161019968)The count is 4

(thread 124635135841856)The count is 4(thread 124635177805376)--------------------(thread 124635169412672)--------------------(thread 124634582218304)--------------------(thread 124634573825600)The count is 4(thread 124635161019968)--------------------(thread 124635177805376)

(thread 124634573825600)--------------------Finishing up

(thread 124635161019968)
(thread 124634582218304)
(thread 124635169412672)
(thread 124634573825600)
(thread 124635186198080)The count is 5

### One possible execution order

| time | Thread 0 | Thread 1 ... | Thread 9 | counter |
| ---- | -------- | -------- | -------- | ------- |
| 1  | `oldcnt = counter` → 0   | | | 0 |
| 2  | `counter = oldcnt+1` | | | **1** |
| 3  | `print(count)`       | | | 1 |
| 4  | `print(---)`         | | | 1 |
| 5  | | `oldcnt = counter` → 1   | | 1 |
| 6  | | `counter = oldcnt+1` | | **2** |
| 7  | | `print(count)`       | | 2 |
| 8  | | `print(---)`         | | 2 |
| 9  | | | `oldcnt = counter` → 2   | 2 |
| 10 | | | `counter = oldcnt+1` | **3** |
| 11 | | | `print(count)`       | 3 |
| 12 | | | `print(---)`         | 3 |

### Another possible execution order

| time | Thread 0 | Thread 1 | Thread 9 | counter |
| ---- | -------- | -------- | -------- | ------- |
| 1  | `oldcnt = counter` → 0 | | | 0 |
| 2  | *(interrupted)* | `oldcnt = counter` → 0 | | 0 |
| 3  | | *(interrupted)* | `oldcnt = counter` → 0 | 0 |
| 4  | | | `counter = oldcnt+1` | **1** |
| 5  | | | `print(count)` | 1 |
| 6  | | | `print(--)` | 1 |
| 7  | | `counter = oldcnt+1` | | **1** |
| 8  | | `print(count)` | | 1 |
| 9  | | `print(--)` | | 1 |
| 10 | `counter = oldcnt+1` | | | **1** |
| 11 | `print(count)` | | | 1 |
| 12 | `print(--)` | | | 1 |

All three threads read `0`, all write `1`, and the counter never advances past 1 — three increments collapsed into one.

## Raymond's rules

> RR 1001
> 
> One category of sequencing problems is to make sure that step A and step B happen sequentially.
> 
> The solution is to put both in the same thread where all actions proceed sequentially.


> RR 1002
> 
> To implement a "barrier" that waits for parallel threads to complete, just `join()` all of the threads.

> RR 1003
>
> You can't wait on daemon threads to complete (they are infinite loops). Instead, you join() on the queue itself.
>
> It waits until all teh requested tasks are marked as being done.'


> RR 1004
>
> Sometimes you need a global variable to communicate between functions. G
>
> It waits until all teh requested tasks are marked as being done.


> RR 1005
>
> Never try to kill a thread from something external to that thread.
>
> You never know if that thread is holding a lock. Python doesn't provide a direct mechanism to kill threads
> externally; however it is still possible.
>
> This is a recipe for deadlock.


## Atomic message queues

In [12]:
import threading, random, queue, time

FUZZ = False

def fuzz():
    if FUZZ:
        time.sleep(random.random())

counter = 0

counter_queue = queue.Queue()

def counter_manager():
    global counter
    while True:
        increment = counter_queue.get()
        fuzz()
        oldcnt = counter
        fuzz()
        counter = oldcnt + increment
        fuzz()
        print_queue.put([
            'The count is %d' % counter,
            '---------------'
            ])
        fuzz()
        counter_queue.task_done()

t = threading.Thread(target=counter_manager)
t.daemon = True
t.start()
del t

###################################################

print_queue = queue.Queue()

def print_manager():
    
    while True:
        job = print_queue.get()
        fuzz()
        for line in job:
            print(line, end='')
            fuzz()
            print()
            fuzz()
        print_queue.task_done()
        fuzz()

t = threading.Thread(target=print_manager)
t.daemon = True
t.start()
del t

###################################################

def worker():
    counter_queue.put(1)
    fuzz()

print_queue.put(['Starting up'])
fuzz()

worker_threads = []
for i in range(10):
    t = threading.Thread(target=worker)
    worker_threads.append(t)
    t.start()
    fuzz()

for t in worker_threads:
    fuzz()
    t.join()

counter_queue.join()
fuzz()
print_queue.put(['Finishing up'])
fuzz()
print_queue.join()
fuzz()

Starting up
The count is 1
---------------
The count is 2
---------------
The count is 3
---------------
The count is 4
---------------
The count is 5
---------------
The count is 6
---------------
The count is 7
---------------
The count is 8
---------------
The count is 9
---------------
The count is 10
---------------
Finishing up


In [13]:
import threading, random, queue, time

counter = 0
counter_queue = queue.Queue()

def counter_manager():
    global counter
    while True:
        increment = counter_queue.get()
        counter += 1
        print_queue.put([
            'The count is %d' % counter,
            '---------------'
            ])
        counter_queue.task_done()

t = threading.Thread(target=counter_manager)
t.daemon = True
t.start()
del t

###################################################

print_queue = queue.Queue()

def print_manager():
    
    while True:
        job = print_queue.get()
        for line in job:
            print(line, end='')
            print()
        print_queue.task_done()

t = threading.Thread(target=print_manager)
t.daemon = True
t.start()
del t

###################################################

def worker():
    counter_queue.put(1)

print_queue.put(['Starting up'])
worker_threads = []
for i in range(10):
    t = threading.Thread(target=worker)
    worker_threads.append(t)
    t.start()

for t in worker_threads:
    t.join()

counter_queue.join()
print_queue.put(['Finishing up'])
print_queue.join()

Starting up
The count is 1
---------------
The count is 2
---------------
The count is 3
---------------
The count is 4
---------------
The count is 5
---------------
The count is 6
---------------
The count is 7
---------------
The count is 8
---------------
The count is 9
---------------
The count is 10
---------------
Finishing up


## Locks

Use locks to control when sequence breaks occur so that the state of your program is always correct.

The code below has a bug - can anyone spot it?

In [14]:
import threading, time, random

FUZZ = True
counter = 0

counter_lock = threading.Lock()
printer_lock = threading.Lock()

def fuzz():
    if FUZZ:
        time.sleep(random.random())

def worker():
    'My job is to increment the counter and print the current count'
    global counter
    with printer_lock:
        fuzz()
        oldcnt = counter
        fuzz()
        counter = oldcnt + 1
        fuzz()
        with counter_lock:
            print('The count is %d' % counter, end='')
            fuzz()
            print()
            fuzz()
            print('--------------------', end='')
            fuzz()
            print()
            fuzz()


with printer_lock:
    print('Starting up')

worker_threads = []
for i in range(10):
    t = threading.Thread(target=worker)
    worker_threads.append(t)
    t.start()

for t in worker_threads:
    t.join()

with printer_lock:
    print('Finishing up')

Starting up
(thread 124635135841856)--------------------
(thread 124635186198080)--------------------(thread 124635186198080)
(thread 124635135841856)
The count is 1
--------------------
The count is 2
--------------------
The count is 3
--------------------
The count is 4
--------------------
The count is 5
--------------------
The count is 6
--------------------
The count is 7
--------------------
The count is 8
--------------------
The count is 9
--------------------
The count is 10
--------------------
Finishing up


In [15]:
def workerElegantAndBuggy():
        '''
        The previous implementation was over-engineered.  This implementation is much cleaner and elegant. 
          - the next person that needs to maintain your code
        '''
        oldcnt = counter
        fuzz()
        counter = oldcnt + 1
        fuzz()
        print('The count is %d' % counter, end='')
        fuzz()
        print()
        fuzz()
        print('--------------------', end='')
        fuzz()
        print()
        fuzz()


* Locks work when they're used.
  * Will people always use them?
* Locks work when they're used correctly
  * Will people always use correctly?
* Locks can be used correctly in a way that removes some or all of the benefits of concurrency. 

# Multiprocessing

In [16]:
import urllib.request

sites = [
    'https://www.duckduckgo.com',
    'https://www.python.org',
    'https://www.perl.org',
    'https://www.erlang.org',
    'https://www.haskell.org',
    'https://go.dev',
    'https://rust-lang.org',
    'https://www.typescriptlang.org',
    'https://nodejs.org',
    'https://developer.mozilla.org',
    'https://www.github.com',
    'https://www.vim.org',
    'https://www.gnu.org/software/emacs'
]

for url in sites:
    with urllib.request.urlopen(url) as u:
        page = u.read()
        print(url, len(page))

https://www.duckduckgo.com 160825
https://www.python.org 11445
https://www.perl.org 12371
https://www.erlang.org 23344
https://www.haskell.org 64880
https://go.dev 64029
https://rust-lang.org 18601
https://www.typescriptlang.org 259244
https://nodejs.org 462337


KeyboardInterrupt: 

In [ ]:
from multiprocessing.pool import ThreadPool as Pool

def sitesize(url):
    with urllib.request.urlopen(url) as u:
        page = u.read()
        return url, len(page)

pool = Pool(10)
for result in pool.imap_unordered(sitesize, sites):
    print(result)

### notes
[`imap_unordered`](https://docs.python.org/3/library/multiprocessing.html#multiprocessing.pool.Pool.imap_unordered) is used to improve responsiveness.

# Async

## Async vs Threads

### Threads

Threads switch preemptively.  Convenient because you don't need to add explicit code to cause a task switch

The cost is that you have to assume a switch can happen at any time. Critical sections have to be guarded with locks

Threads cost CPU

### Async

Async switches cooperatively, so you DO need to add explicit code "yield" or "await" to cause a task switch.

Now you control when task switches occur, so locks and other synchronization are no longer needed.

The cost of task switches if very low, so async is very cheap.


What's the cost? You'll need a non-blocking version of just about everything you do. 

Accordingly, the async world has a huge ecosystem of support tools. 

This increases the learning curve, and the cost to turn sequential code into concurrent code.

In [ ]:
import asyncio
import random

FUZZ = True
counter = 0
lock = asyncio.Lock()

async def fuzz():
    if FUZZ:
        await asyncio.sleep(random.random())

async def worker():
    global counter
    await fuzz()

    async with lock:
        oldcnt = counter
        counter = oldcnt + 1
        output = f'The count is {counter}\n--------------------\n'

    await fuzz()
    print(output, end='')
    await fuzz()

async def main():
    print('Starting up')
    await fuzz()
    tasks = [asyncio.create_task(worker()) for _ in range(10)]
    await asyncio.gather(*tasks)
    print('Finishing up')
    print(f'At the end, the counter is {counter}')

await main()


# Nitty gritty details

## Global Interpreter Lock (python-specific)

No more than one thread can run at a time.

* Not an issue for I/O bound applications.
* A significant issue for CPU-bound applications.

## Hazards of multiprocessing

Use "thin channel communication"

* Do not make too many trips back and forth.
* Do significant work on each trip.


### Do not make too many trips back and forth

In [ ]:
# BAD!
# Split each webpage into many pieces and compute on each one.
# If the iterable to map is very large, it suggests you're making too many trips
def sitesize(url, start):
    req = urllib.request.Request()
    req.add_header('Range:%d-%d' % (start, start+1000))
    with urllib.request.urlopen(url, req) as u:
        block = u.read()
        return url, len(block)

### Do significant work on each trip

In [ ]:
# BAD! 
# Not doing enough work relative to the travel time
# Once you get a process, be sure to do enough work to make the trip worthwhile
def sitesize(url, results):
    with urllib.request.urlopen(url) as u:
        while True:
            line = u.readline()
            results.put((url,len(line))

### Do not send or receive a lot of data

In [ ]:
# BAD! 
# Returns too much data
def sitesize(url):
    with urllib.request.urlopen(url) as u:
        page = u.read()
        return url, page
